In [1]:
import pandas as pd
import numpy as np
customers = pd.read_csv("../../data/raw/customers.csv")
order_items = pd.read_csv("../../data/raw/order_items.csv")
orders = pd.read_csv("../../data/raw/orders.csv")
products = pd.read_csv("../../data/raw/products.csv")

print(customers.head())
print(order_items.head())
print(orders.head())
print(products.head())

   customer_id name gender  age city signup_date
0            1  김수민      F   19   광주  2024-08-14
1            2  김정호      F   32   대구  2025-12-28
2            3  이경수      F   61   성남  2024-08-07
3            4  조영호      F   55   울산  2026-06-08
4            5  이예원      F   19   부산  2024-11-08
   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000
3              4         1           9         3      193000
4              5         2          72         4      189000
   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-02           card    completed
1         2           77  2025-09-17      naver_pay    cancelled
2         3          138  2026-01-14  bank_transfer    cancelled
3         4           57  2026-03-27      kakao_pay    cancelled
4         5    

In [2]:
# 원본 구조 Evidence(이력)를 만든다.

raw_data = {
    "customers" : customers,
    "order_items" : order_items,  
    "orders": orders, 
    "products": products
}

print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-08-14
1              2  김정호      F   32   대구  2025-12-28
2              3  이경수      F   61   성남  2024-08-07
3              4  조영호      F   55   울산  2026-06-08
4              5  이예원      F   19   부산  2024-11-08
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2026-02-17
146          147  이정남      M   19   부산  2025-04-08
147          148  오도현      M   29   고양  2026-08-10
148          149  김정자      M   20   부산  2024-12-14
149          150  조미영      M   40   대전  2026-01-29

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [3]:
summary_list = []
for name, frame in raw_data.items():
    # print(name)
    info = {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "duplicated_rows" : int(frame.duplicated().sum()),
    }
    summary_list.append(info)

summary_list

[{'dataset': 'customers',
  'rows': 150,
  'columns': 6,
  'missing_values': 0,
  'duplicated_rows': 0},
 {'dataset': 'order_items',
  'rows': 764,
  'columns': 5,
  'missing_values': 0,
  'duplicated_rows': 0},
 {'dataset': 'orders',
  'rows': 300,
  'columns': 5,
  'missing_values': 0,
  'duplicated_rows': 0},
 {'dataset': 'products',
  'rows': 100,
  'columns': 4,
  'missing_values': 0,
  'duplicated_rows': 0}]

In [4]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)

relationship_checks = validate_relationships(
    processed_data
)


In [5]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(raw_data, processed_data)



In [6]:
preprocessing_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [7]:
print(raw_data["order_items"].columns)
print(processed_data["order_items"].columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'], dtype='str')
Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


In [8]:
print(raw_data["orders"].columns)
print(processed_data["orders"].columns)

Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')


In [9]:
print(processed_data["orders"].head())

   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-07-02           card    completed     2026-07   
1         2           77 2025-09-17      naver_pay    cancelled     2025-09   
2         3          138 2026-01-14  bank_transfer    cancelled     2026-01   
3         4           57 2026-03-27      kakao_pay    cancelled     2026-03   
4         5          125 2026-02-15           card    cancelled     2026-02   

  order_dayofweek  
0        Thursday  
1       Wednesday  
2       Wednesday  
3          Friday  
4          Sunday  


In [10]:
print(processed_data["orders"]["order_dayofweek"].value_counts())

order_dayofweek
Saturday     50
Tuesday      47
Monday       47
Wednesday    44
Sunday       43
Friday       35
Thursday     34
Name: count, dtype: int64


In [11]:
key_map = {
    "customers" : "customer_id",
    "products" : "product_id",
    "orders" : "order_id",
    "order_items" : "order_id"
}

In [12]:
kp_checks = []

for dataset, key in key_map.items():
    frame = processed_data[dataset]
    missing_count = int(frame[key].isna().sum())
    duplicated_count = int(frame[key].duplicated().sum())
    status = ""
    if missing_count == 0 and duplicated_count == 0:
        status = "PASS"
    else:
        status = "FAIL"
    kp_checks.append({
        "dataset": dataset, 
        "key": key, 
        "missing_count": missing_count, 
        "duplicated_count": duplicated_count, 
        "status": status,
    })

In [13]:
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)


In [14]:
print(len(order_items))
print(len(order_sales))
order_sales["_merge"].value_counts(dropna=False)

764
764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [15]:
expected_line_total = order_items["quantity"] * order_items["unit_price"]

In [16]:
if "line_total" not in order_items.columns:
    order_items["line_total"] = expected_line_total

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
 5   line_total     764 non-null    int64
dtypes: int64(6)
memory usage: 35.9 KB


In [17]:
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")
]

print(completed_order_sales.head())
print(completed_order_sales["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0               1         1         100         3      102000          123   
1               2         1          87         5       25000          123   
2               3         1           7         3      142000          123   
3               4         1           9         3      193000          123   
12             13         6          83         3       24000           87   

    order_date order_status _merge  
0   2026-07-02    completed   both  
1   2026-07-02    completed   both  
2   2026-07-02    completed   both  
3   2026-07-02    completed   both  
12  2026-05-16    completed   both  
order_status
completed    474
Name: count, dtype: int64


In [18]:
completed_order_sales["line_total"] = ( completed_order_sales["quantity"] * completed_order_sales["unit_price"])

print("전체 주문 수:", order_sales["order_item_id"].value_counts().sum())
print("전체 주문 건수(completed):", len(completed_order_sales))
print("전체 주문 건수(completed 외): ", len(order_sales) - len(completed_order_sales))

order_sales["order_status"].value_counts()

completed_order_sales.columns


전체 주문 수: 764
전체 주문 건수(completed): 474
전체 주문 건수(completed 외):  290


Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'customer_id', 'order_date', 'order_status', '_merge', 'line_total'],
      dtype='str')

In [19]:
order_status_sales = (
    completed_order_sales
    .groupby("order_status", as_index = False)
    .agg(
        total_quantity = ("quantity", "sum"),
        total_sales = ("line_total", "sum")
    )
    .sort_values("total_sales", ascending = False)
)
print(completed_order_sales.columns)
print(products.head())


Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'customer_id', 'order_date', 'order_status', '_merge', 'line_total'],
      dtype='str')
   product_id product_name category   price
0           1  전자기기 상품 001     전자기기  160000
1           2    도서 상품 002       도서   34000
2           3  전자기기 상품 003     전자기기  152000
3           4  생활용품 상품 004     생활용품   70000
4           5    식품 상품 005       식품  186000
